# SQLAlchemy Core

## Overview
**SQLAlchemy Core** is the SQL expression layer of SQLAlchemy — a thin but powerful wrapper that manages connections, pools, and dialects while keeping SQL close to hand.

**SQLAlchemy 2.0 key changes from 1.x:**
- `engine.execute()` removed — always use `with engine.connect() as conn:`
- `conn.execute(text("..."))` requires `text()` wrapper for raw SQL strings
- `engine.begin()` auto-commits on exit; `engine.connect()` requires explicit `conn.commit()`
- Results are `CursorResult` with `.mappings()`, `.tuples()`, `.all()`, `.first()`

**SQLAlchemy layers:**

| Layer | What it does | Use when |
|---|---|---|
| **Core** | SQL expression language; manages connections | ETL, data engineering, complex queries |
| **ORM** | Maps Python classes to tables; session management | Application development, CRUD APIs |

---

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=False)

with engine.begin() as conn:   # engine.begin() auto-commits
    conn.execute(text("""
        CREATE TABLE patients (
            patient_id TEXT PRIMARY KEY, first_name TEXT NOT NULL,
            last_name TEXT NOT NULL, province TEXT NOT NULL,
            dob TEXT NOT NULL, sex TEXT NOT NULL
        )
    """))
    conn.execute(text("""
        CREATE TABLE encounters (
            enc_id INTEGER PRIMARY KEY, patient_id TEXT NOT NULL,
            enc_date TEXT NOT NULL, enc_type TEXT NOT NULL, cost REAL DEFAULT 0.0
        )
    """))
    conn.execute(text("""
        INSERT INTO patients VALUES
            ('P001','Aroha','Ngata','NB','1985-03-15','F'),
            ('P002','Liam','Chen','ON','1990-07-22','M'),
            ('P003','Fatima','Rashid','BC','1978-11-05','F'),
            ('P004','James','MacLeod','NB','2001-01-30','M'),
            ('P005','Maria','Santos','QC','1995-06-18','F')
    """))
    conn.execute(text("""
        INSERT INTO encounters VALUES
            (1,'P001','2023-04-10','outpatient', 250.0),
            (2,'P001','2023-07-15','inpatient', 4800.0),
            (3,'P002','2023-03-01','outpatient',  80.0),
            (4,'P003','2023-09-20','specialist',  450.0),
            (5,'P004','2023-11-05','outpatient',  120.0),
            (6,'P005','2023-06-12','inpatient',  3200.0)
    """))

print("Engine:", engine.url)
with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM patients")).scalar()
    print(f"Patients: {n}")


---
## Executing queries and reading results

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///:memory:", echo=False)
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE patients (patient_id TEXT PRIMARY KEY, first_name TEXT, last_name TEXT, province TEXT, dob TEXT, sex TEXT)"))
    conn.execute(text("CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id TEXT, enc_date TEXT, enc_type TEXT, cost REAL DEFAULT 0.0)"))
    conn.execute(text("INSERT INTO patients VALUES ('P001','Aroha','Ngata','NB','1985-03-15','F'),('P002','Liam','Chen','ON','1990-07-22','M'),('P003','Fatima','Rashid','BC','1978-11-05','F'),('P004','James','MacLeod','NB','2001-01-30','M'),('P005','Maria','Santos','QC','1995-06-18','F')"))
    conn.execute(text("INSERT INTO encounters VALUES (1,'P001','2023-04-10','outpatient',250.0),(2,'P001','2023-07-15','inpatient',4800.0),(3,'P002','2023-03-01','outpatient',80.0),(4,'P003','2023-09-20','specialist',450.0),(5,'P004','2023-11-05','outpatient',120.0),(6,'P005','2023-06-12','inpatient',3200.0)"))

with engine.connect() as conn:
    # Named :param_name bound parameters
    result = conn.execute(
        text("SELECT patient_id, first_name, province FROM patients WHERE province = :prov"),
        {"prov": "NB"}
    )
    print("NB patients:")
    for row in result:
        print(f"  {row.patient_id}  {row.first_name}  — {row.province}")

    # Result access patterns
    result = conn.execute(text("SELECT * FROM encounters ORDER BY cost DESC"))
    first  = result.first()
    print(f"\nHighest cost encounter: enc_id={first.enc_id}, cost=${first.cost:,.2f}")

    result = conn.execute(text("SELECT * FROM patients"))
    maps   = result.mappings().all()
    print(f"\nFirst patient as mapping: {dict(maps[0])}")

    # scalar() for single-value queries
    total_cost = conn.execute(
        text("SELECT SUM(cost) FROM encounters")
    ).scalar()
    print(f"\nTotal encounter cost: ${total_cost:,.2f}")

    # Pagination with fetchmany
    print("\nPatients in batches of 2:")
    result = conn.execute(text("SELECT patient_id, first_name FROM patients ORDER BY patient_id"))
    while True:
        batch = result.fetchmany(2)
        if not batch:
            break
        print(f"  {[r.patient_id for r in batch]}")


---
## Table constructs and programmatic SQL

In [ ]:
from sqlalchemy import (create_engine, MetaData, Table, Column,
                        String, Integer, Float, select, insert, func)

engine = create_engine("sqlite:///:memory:", echo=False)
meta   = MetaData()

patients_t = Table("patients", meta,
    Column("patient_id", String, primary_key=True),
    Column("first_name", String, nullable=False),
    Column("last_name",  String, nullable=False),
    Column("province",   String, nullable=False),
    Column("sex",        String, nullable=False),
)
encounters_t = Table("encounters", meta,
    Column("enc_id",     Integer, primary_key=True, autoincrement=True),
    Column("patient_id", String,  nullable=False),
    Column("enc_date",   String,  nullable=False),
    Column("enc_type",   String,  nullable=False),
    Column("cost",       Float,   default=0.0),
)
meta.create_all(engine)

with engine.begin() as conn:
    conn.execute(insert(patients_t), [
        {"patient_id":"P001","first_name":"Aroha","last_name":"Ngata","province":"NB","sex":"F"},
        {"patient_id":"P002","first_name":"Liam","last_name":"Chen","province":"ON","sex":"M"},
        {"patient_id":"P003","first_name":"Fatima","last_name":"Rashid","province":"BC","sex":"F"},
    ])
    conn.execute(insert(encounters_t), [
        {"patient_id":"P001","enc_date":"2023-04-10","enc_type":"outpatient","cost":250.0},
        {"patient_id":"P001","enc_date":"2023-07-15","enc_type":"inpatient","cost":4800.0},
        {"patient_id":"P002","enc_date":"2023-03-01","enc_type":"outpatient","cost":80.0},
    ])

with engine.connect() as conn:
    stmt = (
        select(
            patients_t.c.patient_id,
            patients_t.c.first_name,
            func.count(encounters_t.c.enc_id).label("enc_count"),
            func.sum(encounters_t.c.cost).label("total_cost")
        )
        .join(encounters_t, patients_t.c.patient_id == encounters_t.c.patient_id)
        .group_by(patients_t.c.patient_id, patients_t.c.first_name)
        .order_by(func.sum(encounters_t.c.cost).desc())
    )
    print("Compiled SQL:")
    print(" ", str(stmt.compile(engine, compile_kwargs={"literal_binds": True})))
    print()
    print("Results:")
    for row in conn.execute(stmt):
        print(f"  {row.patient_id}  {row.first_name:<8s}  "
              f"encounters={row.enc_count}  total=${row.total_cost:,.2f}")


---
## Common Pitfalls

**1. Using raw strings in execute() without text() in SQLAlchemy 2.0**
`conn.execute('SELECT * FROM patients')` raises `ObjectNotExecutableError` in SQLAlchemy 2.0. Raw SQL must be wrapped: `conn.execute(text('SELECT * FROM patients'))`. This is a deliberate change from 1.x to prevent accidental injection vulnerabilities.

**2. Not committing with engine.connect()**
`with engine.connect() as conn:` does NOT auto-commit. Call `conn.commit()` explicitly, or use `with engine.begin() as conn:` which auto-commits on successful block exit and rolls back on exception.

**3. echo=True left on in production**
`create_engine(..., echo=True)` logs every SQL statement. In production this floods logs, reveals table structure, and adds I/O overhead. Always `echo=False` in production.

**4. Reusing a result object after the connection closes**
A `CursorResult` is only valid while the connection is open. Call `result.all()` inside the `with engine.connect()` block to materialise rows before the connection closes.

**5. Mixing SQLAlchemy Core and ORM transactions on the same session**
Executing raw `text()` through an ORM Session alongside ORM statements can cause the Session's identity map to become inconsistent. Use Core or ORM exclusively within one transaction.

**6. Not disposing the engine on application shutdown**
`engine.dispose()` closes all pooled connections. Long-running processes that create engines without disposing them exhaust database connection limits. Call `engine.dispose()` at shutdown.


---
*sql_methods_library - Samantha McGarrigle*